In [1]:
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset
import tqdm
import torch
import random

import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import copy

from torch.utils.data import DataLoader

from source.f_dataloader_withpair import Vocab, Attribute#, DietDataset

from evaluation.eval_loader import EditDataset

test_datapath = "/home/siun97/Diet/KDD2025/AGGBERT/evaluation/test_for_rfm.csv"
# Model: 
model_path = "/home/siun97/Diet/KDD2025/AGGBERT/trained_models/finetuned/model.best"
finetuned_model = torch.load(model_path, map_location = "cuda:1")

In [4]:
cuda_condition = torch.cuda.is_available()
device = torch.device("cuda:1" if cuda_condition else "cpu")

from metric import new_mispos_score
import pickle

morning_comb = pickle.load(open('./evaluation/morning_comb.pkl', 'rb')) 
lunch_comb = pickle.load(open('./evaluation/lunch_comb.pkl', 'rb')) 
dinner_comb = pickle.load(open('./evaluation/dinner_comb.pkl', 'rb')) 
food_columns_dict = pickle.load(open('./evaluation/food_columns_dict.pkl', 'rb'))
allergy_ref = pd.read_csv('./data/allergy_db_updated.csv', encoding = 'cp949')

'''
0. data 불러오기
'''
corpus_path = "./data/train_diet_temp.csv"
vocab_path = "./data/entity_vocab_4067.csv"
menu_path = "./data/total_nutri_data.csv"

vocab = Vocab(corpus_path = corpus_path, vocab_path = vocab_path, menu_path=menu_path, load_vocab=True, expand = False)

allergy_external_path = "./data/allergy_db_updated.csv"
ingredi_external_path = "./data/total_ingredient_data.csv"

attribute = Attribute(vocab, allergy_external_path = allergy_external_path, 
                        ingre_external_path = ingredi_external_path, 
                        attribute_type = "ingredient")

# DataParallel을 사용하여 저장된 경우, module 속성에 접근
if isinstance(finetuned_model , torch.nn.DataParallel):
    finetuned_model = finetuned_model.module

# 디바이스로 이동
finetuned_model = finetuned_model.to(device)

# Graph
graph_emb_path = "/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/cos_sim_matrix.pt"

from source.sampler import TemperatureSampler, NucleusSampler, GreedySampler, TopKSampler
decoding = GreedySampler()

def get_allergy_combinations(allergy_combination_reference_path):

    allergy_combination_df = pd.read_csv(allergy_combination_reference_path, sep=',', header=None)
    allergy_list = allergy_combination_df.to_numpy().tolist()
    temp_allergy_list = []
    for allergy_combination in allergy_list:
        temp = allergy_combination[0].split(',')
        temp_allergy_list.append(temp)
    return temp_allergy_list

def apply_repetition_penalty(logits, generated_tokens, penalty):
    
    for batch_idx, tokens in enumerate(generated_tokens):
        for token in tokens: 
            if logits[batch_idx, token] > 0:
                logits[batch_idx, token] /= penalty
                
            else:
                logits[batch_idx, token] *= penalty
    
    return logits

def decoding_process(allergy_condition, max_len=13, graph_emb_use = True, graph_emb_path = graph_emb_path, attribute_type = "ingredient"):
    test_data = EditDataset(test_datapath, vocab, attribute, allergy_condition, attribute_type)
    test_data_loader = DataLoader(test_data, 128, num_workers=8, shuffle=False)
    
    if graph_emb_use:
        #graph embedding_setting
        graph_emb = torch.load(graph_emb_path + ', '.join(allergy_condition) +'.pt')
        graph_emb_norm = (graph_emb - graph_emb.min()) / (graph_emb.max() - graph_emb.min())
        
        new_emb = torch.zeros((vocab.vocab_size, vocab.vocab_size))
        # special tokens 
        new_emb[:3,:] = 1.0
        new_emb[3:, 3:] = graph_emb_norm
        diag_idx = torch.arange(new_emb.size(0))
        new_emb[diag_idx, diag_idx] = 0
        new_emb = new_emb.to(device)
    
    total_seq = []
    for _, data in enumerate(test_data_loader):
        data = {key: value.to(device) for key, value in data.items()}
        
        encoder_input = data['encoder_input']
        decoder_target = data['decoder_target']
        attribute_mask = data['changed_attribute']
        allergy_token_map = data['allergy_token_map']
        
        with torch.no_grad():
            finetuned_model.eval()
            
            src_mask = finetuned_model.make_src_mask(encoder_input)
            encoder_output = finetuned_model.encoder(encoder_input, src_mask, attribute_mask)
            generated = torch.full((encoder_input.size(0), 1), fill_value=0, dtype=torch.long,  device=device)
            
                #allergy_token_prob_mask = allergy_token_map * -1e8
            
            #sequence_probs = []
            
            for _ in range(max_len):
                trg_mask = finetuned_model.make_trg_mask(generated)
                cross_mask = finetuned_model.make_cross_attn_mask(generated, encoder_input)
                
                output = finetuned_model.decoder(generated, trg_mask, encoder_output, cross_mask)
                output = finetuned_model.mask_lm(output)
                
                next_token_logits = output[:, -1, :]
                
                if graph_emb_use:
                    masked_logits = apply_repetition_penalty(next_token_logits, generated, 1.4)
                    new_token_logits = torch.exp(masked_logits)
                    next_token = decoding(new_token_logits).unsqueeze(-1)
                    
                    new_prob = new_emb[next_token.squeeze(1), :] * new_token_logits
                    
                    #new_prob = apply_repetition_penalty(new_prob, generated, 1.4)
                    
                    graph_embed_next_token = decoding(new_prob).unsqueeze(-1)
                    
                    allergy_tokens_idx = np.where(allergy_token_map.detach().cpu()[torch.arange(encoder_input.size(0)), 
                                                                              next_token.squeeze(1).detach().cpu()] == 1)
                    
                    next_token[allergy_tokens_idx] = graph_embed_next_token[allergy_tokens_idx]
                    
                else:
                    #masked_logits = next_token_logits #+ allergy_token_prob_mask
                    masked_logits = apply_repetition_penalty(next_token_logits, generated, 1.4)
                    next_token = decoding(masked_logits).unsqueeze(-1)
                
                generated = torch.cat((generated, next_token), dim=1)
                #sequence_probs.append(next_token_logits)
                
                
                if generated.size(1) == max_len+1:
                    break
                
        total_seq.append(generated[:, 1:])
         
    return torch.cat(total_seq, axis = 0)


import torch.nn.functional as F
import torch.utils.data as tud

def beam_search_decoding_process(allergy_condition, max_len=13, beam_size=3, graph_emb_path = graph_emb_path, graph_emb_use = False, attribute_type = "ingredient"):
    test_data = EditDataset(test_datapath, vocab, attribute, allergy_condition, attribute_type)
    test_data_loader = DataLoader(test_data, 128, num_workers=8, shuffle=False)
    
    total_seq = []
    total_score = []
    
    if graph_emb_use:
        total_graph_emb = torch.load(graph_emb_path)
        graph_emb_list_path = "/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/31_allergy_list.txt"
        with open(graph_emb_list_path, "r", encoding = 'utf-8') as f:
            graph_emb_list = f.readlines()
        graph_emb_list = [line.strip() for line in graph_emb_list]

        graph_idx = graph_emb_list.index((", ").join(allergy_condition))

        graph_emb = total_graph_emb[graph_idx]
        new_emb = torch.zeros((vocab.vocab_size, vocab.vocab_size))
        # special tokens 
        new_emb[:3,:] = 1.0
        new_emb[3:, 3:] = graph_emb
        diag_idx = torch.arange(new_emb.size(0))
        new_emb[diag_idx, diag_idx] = 0
        new_emb = (new_emb+1)/2
        new_emb = new_emb.to(device)
    
    

    for _, data in enumerate(test_data_loader):
        data = {key: value.to(device) for key, value in data.items()}
        
        encoder_input = data['encoder_input']
        attribute_mask = data['changed_attribute']
        allergy_token_map = data['allergy_token_map']
        decoder_target = data['decoder_target']
        
        with torch.no_grad():
            finetuned_model.eval()
            
            src_mask = finetuned_model.make_src_mask(encoder_input)
            encoder_output = finetuned_model.encoder(encoder_input, src_mask, attribute_mask)
            
            #first decoding
            trg_mask = finetuned_model.make_trg_mask(decoder_target)
            cross_mask = finetuned_model.make_cross_attn_mask(decoder_target, encoder_input)
            output = finetuned_model.decoder(decoder_target, trg_mask, encoder_output, cross_mask)
            output = torch.exp(finetuned_model.mask_lm(output))
            
            next_prob, idx = output.squeeze().topk(k =beam_size, axis = -1)
            # select initial beam nodes
            seq = decoder_target.repeat((beam_size, 1, 1)).transpose(0,1).flatten(end_dim = -2)
            # beam size 만큼 반복 (batch*beam_size, token_num)
            
            next_token = idx.reshape(-1,1)
            
            if graph_emb_use:
                next_prob = next_prob.reshape(-1,1)
                
                graph_embed_prob = output.repeat_interleave(beam_size, dim= 1).view(-1, new_emb.size(1)) * new_emb[next_token.squeeze(1), :]
                allergy_tokens_idx = np.where(allergy_token_map.repeat_interleave(beam_size, dim=0).detach().cpu()[torch.arange(encoder_input.size(0)*beam_size), 
                                                            next_token.squeeze(1).detach().cpu()] == 1)
                graph_embed_next_prob, graph_embed_next_token = graph_embed_prob.topk(k=1, axis = -1)
                
                next_token[allergy_tokens_idx] = graph_embed_next_token[allergy_tokens_idx]
                next_prob[allergy_tokens_idx] = graph_embed_next_prob[allergy_tokens_idx]
                
                next_prob = next_prob.reshape(encoder_input.size(0), beam_size)
            
            seq = torch.cat((seq, next_token), axis = -1)
            
            for _ in range(max_len-1):
                
                loader = tud.DataLoader(seq, batch_size = encoder_output.size(0))
                next_probabilities = []

                for _, a in enumerate(loader):
                    trg_mask = finetuned_model.make_trg_mask(a)
                    cross_mask = finetuned_model.make_cross_attn_mask(a, encoder_input)
                    output = finetuned_model.decoder(a, trg_mask, encoder_output, cross_mask)
                    next_token_logits = finetuned_model.mask_lm(output)[:, -1, :]
                    logits = apply_repetition_penalty(next_token_logits, a, 1.4)
                    next_probabilities.append(torch.exp(logits))
                    
                next_probabilities = torch.cat(next_probabilities, axis = 0)
                next_probabilities = next_probabilities.reshape(-1, beam_size, next_probabilities.shape[-1])
                #[batch, beam_size, vocab_size]
                
                new_prob, new_idx = next_probabilities.squeeze().topk(k =beam_size, axis = -1)
                
                if graph_emb_use:
                    new_idx = new_idx.reshape(-1, 1)
                    new_prob = new_prob.reshape(-1,1)
                    
                    graph_embed_prob = next_probabilities.repeat_interleave(beam_size, dim= 1).view(-1, new_emb.size(1)) * new_emb[new_idx.squeeze(1), :]
                    allergy_tokens_idx = np.where(allergy_token_map.repeat_interleave(beam_size*beam_size, dim=0).detach().cpu()[torch.arange(encoder_input.size(0)*beam_size*beam_size), 
                                                            new_idx.squeeze(1).detach().cpu()] == 1)
                    graph_embed_next_prob, graph_embed_next_token = graph_embed_prob.topk(k=1, axis = -1)
                    
                    new_idx[allergy_tokens_idx] = graph_embed_next_token[allergy_tokens_idx]
                    new_prob[allergy_tokens_idx] = graph_embed_next_prob[allergy_tokens_idx]
                    
                    new_prob = new_prob.reshape(encoder_input.size(0), beam_size, beam_size)
                    new_idx = new_idx.reshape(encoder_input.size(0), beam_size, beam_size)
                    
                # 각 beam의 prediction 중에서 3개 (vocab 중에서 3개)
                iter_prob = (next_prob.unsqueeze(-1) * new_prob).view(encoder_output.size(0), -1)
                
                next_prob, total_seq_idx = iter_prob.squeeze().topk(k=beam_size, axis = -1)
                #beam 줄기 중에서 3개
                
                new_seq = torch.cat((seq.repeat((beam_size,1,1)).transpose(0,1).flatten(end_dim = -2).reshape(encoder_output.size(0), beam_size*beam_size, seq.size(1)), new_idx.view(encoder_output.size(0),-1).unsqueeze(-1)), dim = -1)
                seq = torch.gather(new_seq, 1, total_seq_idx.unsqueeze(-1).expand(-1, -1, new_seq.size(-1))).reshape(encoder_output.size(0)*beam_size, -1)

                if seq.size(1) == max_len+1:
                    break
        
        total_seq.append(seq[:, 1:])
        total_score.append(next_prob)
                
    return torch.cat(total_seq, axis = 0), torch.cat(total_score, axis =0)


def allergy_free_test(sample_diet, ref_database, allergy):
    
    allergy_rate = 0
    if ('</s>' in sample_diet) or ('밀' in sample_diet) or ('우유' in sample_diet) or ('난류' in sample_diet) or ('대두' in sample_diet) or ('땅콩+견과류' in sample_diet):
        sample_diet = [token for token in sample_diet if token != '</s>']
    else:
        allergy_exist = [i for i in sample_diet if ref_database[ref_database.name == i][allergy].values[0].any()]
        allergy_rate += len(allergy_exist)
        
    return allergy_rate

def ingre_cosine_sim(input, output, allergy_name, new_emb):
    input_idx = list(map(vocab.convert_tokens_to_ids, input))
    output_idx = list(map(vocab.convert_tokens_to_ids, output))
    
    cosine_sim = torch.mean(new_emb[input_idx, output_idx]).item()

    return cosine_sim

input = pd.read_csv(test_datapath).to_numpy().tolist()

def eval_diet(sequences, allergy_reference, allery_name, food_columns_dict, morning_comb, lunch_comb, dinner_comb, vocab):
    '''
    sequece : 2d list of sequence (string) [num_sequences, seq_len]
    '''
    total_num_duplicates = 0 # 낮을 수록 좋음
    total_mps_score = 0 # 낮을 수록 좋음
    total_ingredient_cosine_sim = 0
    total_success_rate = 0
    
    
    #graph embedding_setting
    total_graph_emb = torch.load("/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/cos_sim_matrix_before_allergy_layer.pt")
    #graph_emb_list_path = "/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/31_allergy_list.txt"
    #with open(graph_emb_list_path, "r", encoding = 'utf-8') as f:
    #    graph_emb_list = f.readlines()
    #graph_emb_list = [line.strip() for line in graph_emb_list]

    #graph_idx = graph_emb_list.index((", ").join(allery_name))

    graph_emb = total_graph_emb#[graph_idx]
    
    new_emb = torch.zeros((vocab.vocab_size, vocab.vocab_size))
    # special tokens 
    #new_emb[:3,:] = 0.0
    new_emb[3:, 3:] = graph_emb
    #diag_idx = torch.arange(new_emb.size(0))
    new_emb[2, 2] = 1.0
   # new_emb = (new_emb+1)/2

    for s_idx, seq in enumerate(sequences):
        # [1] check duplication (composition 3) <- 식단 내에 중복 음식이 있는 가? (없어야 함)
        temp_seq = [token for idx, token in enumerate(seq) if idx not in [0,3,7,8,12] ] # 밥 위치 제외
        if len(temp_seq) != len(set(temp_seq)): # 0,5,12
            total_num_duplicates += 1
        # [2] check location (composition 1) <- 기존 식단 작성 규칙에 위배 되는 가? (0점이어야 함)
        mps_score, _ = new_mispos_score(seq, food_columns_dict, morning_comb, lunch_comb, dinner_comb)
        if mps_score != 0:
            total_mps_score += mps_score/len(seq)
        allergy_rate = allergy_free_test(seq, allergy_reference, allery_name)
        if allergy_rate == 0:
            total_success_rate += 1
        
        
        total_ingredient_cosine_sim += ingre_cosine_sim(input[s_idx], seq, allery_name, new_emb)
        
    
    total_num_duplicates /= len(sequences)
    total_mps_score /= len(sequences)
    total_ingredient_cosine_sim /= len(sequences)
    total_success_rate /= len(sequences)

    return total_num_duplicates, total_mps_score, total_ingredient_cosine_sim, total_success_rate

In [5]:
total_graph_emb = torch.load("/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/before_allergy_layer.pt")

In [6]:
total_graph_emb.shape

torch.Size([4067, 500])

In [7]:
total_graph_emb = torch.load(graph_emb_path)
total_graph_emb.shape

torch.Size([31, 4067, 4067])

In [42]:
graph_emb_list_path = "/home/siun97/Diet/KDD2025/AGGBERT/graph_embedding/31_allergy_list.txt"
with open(graph_emb_list_path, "r", encoding = 'utf-8') as f:
    graph_emb_list = f.readlines()

graph_emb_list = [line.strip() for line in graph_emb_list]

graph_idx = graph_emb_list.index((", ").join(['난류','밀']))

In [43]:
graph_idx

8

In [34]:
finetuned_model

AGGBART(
  (encoder): AGGBART_Encoder(
    (embedding): BARTEmbedding(
      (token): TokenEmbedding(4070, 1024, padding_idx=0)
      (dropout): Dropout(p=0.1, inplace=False)
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (attribute_embedding): AttrEmbedding(3194, 1024)
    (transformer_blocks): ModuleList(
      (0-1): 2 x BartEncoderBlock(
        (attention): MultiHeadedAttention(
          (linear_layers): ModuleList(
            (0-2): 3 x Linear(in_features=1024, out_features=1024, bias=True)
          )
          (output_linear): Linear(in_features=1024, out_features=1024, bias=True)
          (attention): Attention()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (alibi_attention): MultiHeadedAttention(
          (linear_layers): ModuleList(
            (0-2): 3 x Linear(in_features=1024, out_features=1024, bias=True)
          )
          (output_linear): Linear(in_features=1024, out_features=1024, bias=True)
          (att

In [8]:
allergy_conditions = get_allergy_combinations("./data/allergy_candidates.csv")

In [30]:
### BEAM SEARCH ALGORITHM
from tqdm import tqdm

greedy = False
beam_size = 3

allergy_free_diets = {}
for allergy in tqdm(allergy_conditions):
    
    if greedy: 
        seq_candi, seq_prob = beam_search_decoding_process(allergy, beam_size = beam_size, graph_emb_use = False, attribute_type = 'ingredient')
        
        seq_candi = seq_candi.view(-1,beam_size,13)
        _, best_idx = torch.max(seq_prob, dim= 1)
        
        generated = torch.gather(seq_candi, 1, best_idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 13)).squeeze(1)
        final_diet =[list(map(vocab.convert_ids_to_tokens, generated[i].detach().cpu().numpy())) for i in range(len(generated))]
    
    else:
        seq_candi, seq_prob = beam_search_decoding_process(allergy, beam_size = beam_size, graph_emb_use = False, attribute_type = 'ingredient')
        
        seq_candi = seq_candi.view(-1,beam_size,13)
        
        final_diet = []
        for candi in range(len(seq_candi)):
            temp_seqs = [list(map(vocab.convert_ids_to_tokens, seq_candi[candi][i].detach().cpu().numpy())) for i in range(len(seq_candi[candi]))]
            final_diet.append(temp_seqs[torch.argmax(seq_prob[candi])])
    allergy_free_diets[tuple(allergy)] = final_diet  

100%|██████████| 31/31 [1:06:20<00:00, 128.41s/it]


In [31]:
nutrient_df = pd.read_csv("./data/total_nutri_data.csv", engine='python')

### Evaluation

In [ ]:
#1) Only Model

duplicates =[]
mps_scores =[]
ingre_sim_score = []
success_rate =[]

for a in allergy_conditions:
    final_diet = allergy_free_diets[tuple(a)]
    
    total_num_duplicates, total_mps_score, total_ingredient_cosine_sim, total_success_rate = eval_diet(final_diet, nutrient_df, a, food_columns_dict, morning_comb, lunch_comb, dinner_comb, vocab)
    print("allergy: {}".format(a), "total_num_duplicates: {:.4f}".format(total_num_duplicates), "total_mps_score: {:.4f}".format(total_mps_score),"total_ingre_sim_score: {:.4f}".format(total_ingredient_cosine_sim),  'total_success_rate: {:.4f}'.format(total_success_rate))
    
    duplicates.append(total_num_duplicates)
    mps_scores.append(total_mps_score)
    ingre_sim_score.append(total_ingredient_cosine_sim)
    success_rate.append(total_success_rate)

allergy: ['난류'] total_num_duplicates: 0.1140 total_mps_score: 0.0256 total_ingre_sim_score: 0.7360 total_success_rate: 0.9201
allergy: ['우유'] total_num_duplicates: 0.1110 total_mps_score: 0.0246 total_ingre_sim_score: 0.7361 total_success_rate: 0.9882
allergy: ['밀'] total_num_duplicates: 0.1089 total_mps_score: 0.0250 total_ingre_sim_score: 0.7365 total_success_rate: 0.9722
allergy: ['대두'] total_num_duplicates: 0.1167 total_mps_score: 0.0242 total_ingre_sim_score: 0.7361 total_success_rate: 0.9303
allergy: ['땅콩+견과류'] total_num_duplicates: 0.1080 total_mps_score: 0.0251 total_ingre_sim_score: 0.7357 total_success_rate: 0.9979
allergy: ['난류', '우유'] total_num_duplicates: 0.1131 total_mps_score: 0.0263 total_ingre_sim_score: 0.7359 total_success_rate: 0.9116
allergy: ['난류', '밀'] total_num_duplicates: 0.1122 total_mps_score: 0.0276 total_ingre_sim_score: 0.7365 total_success_rate: 0.9041
allergy: ['난류', '대두'] total_num_duplicates: 0.1222 total_mps_score: 0.0250 total_ingre_sim_score: 0.7349

In [33]:
np.mean(duplicates[:5]), np.mean(duplicates[5:15]), np.mean(duplicates[15:25]), np.mean(duplicates[25:30]), np.mean(duplicates[-1])

(0.1117345399698341,
 0.11447963800904978,
 0.11710407239819005,
 0.12018099547511311,
 0.11764705882352941)

In [34]:
np.mean(ingre_sim_score[:5]), np.mean(ingre_sim_score[5:15]), np.mean(ingre_sim_score[15:25]), np.mean(ingre_sim_score[25:30]), np.mean(ingre_sim_score[-1])

(0.7360897903840534,
 0.7359599109980113,
 0.7355405544649198,
 0.7347979067814385,
 0.7340764793832849)

In [35]:
np.mean(mps_scores[:5]), np.mean(mps_scores[5:15]), np.mean(mps_scores[15:25]), np.mean(mps_scores[25:30]), np.mean(mps_scores[-1])

(0.024907761921336622,
 0.025576052906369683,
 0.026023900684534213,
 0.026128321150945627,
 0.02575704838148282)

In [36]:
np.mean(success_rate[:5]), np.mean(success_rate[5:15]), np.mean(success_rate[15:25]), np.mean(success_rate[25:30]), np.mean(success_rate[-1])

(0.9617496229260935,
 0.9292911010558068,
 0.9007843137254902,
 0.8751131221719458,
 0.8509803921568627)

In [37]:
pd.read_csv(test_datapath)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,밤찹쌀팥밥,감자탕,순살닭볶음탕,쌀밥,바지락된장국,참나물볶음,코다리엿장조림,나박김치,수수밥,팥죽,장산적조림,갈치카레튀김,파김치
1,보리밥,미역만두국,양념가지찜,돼지고기양배추덮밥,느타리버섯국,오이토마토샐러드(발사믹드레싱),매콤콩나물무침,나박김치,밤찹쌀팥밥,만두닭개장,empty,애호박볶음,깻잎김치
2,소고기가지덮밥,모듬버섯찌개,우엉채곤약조림,밤콩밥,만두국,애호박건새우볶음,매생이육전,깍두기,수수밥,애호박맑은국,감자채전,팽이버섯나물,깻잎김치
3,잡곡밥,실파콩나물국,달걀새송이장조림,새우편마늘볶음밥,맑은가쓰오부시장국,돼지고기짜장볶음,empty,비트초절이,고구마밥,유부팽이국,감자돼지갈비찜,깻잎새콤무침,나박김치
4,가지채소볶음밥,소고기콩나물국,메추리알고구마조림,밤콩밥,애호박두부젓국,주꾸미채소볶음,미나리초무침,나박김치,밤찹쌀팥밥,감자탕,해물김치전,근대나물무침,총각김치
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3310,옥수수밥,오징어감자국,시금치달걀찜,돼지불고기덮밥,느타리버섯국,동태전,empty,나박김치,수수밥,호박감자수제비국,꽁치조림,진미채간장조림,총각김치
3311,멸치주먹밥,대파무채국,알감자조림,흑미밥,모듬버섯국,소고기토마토조림,청경채초무침,열무김치,오리엔탈비빔쌀국수,대합국,(밀제외)춘권튀김,얼갈이무침,석박지
3312,쌀밥,곰국,무미역생채,브로콜리채소볶음밥,다시마감자국,뼈없는닭갈비,깻잎멸치찜,비트초절이,찰현미밥,모듬버섯찌개,돼지고기김치볶음,우엉채곤약조림,오이소박이김치
3313,찹쌀현미밥,청국장찌개,고추장불고기,소고기가지덮밥,콩나물국,달걀찜(40g),머윗대들깨나물,열무물김치,밤콩밥,김치두부찌개,매콤주꾸미볶음,우엉채곤약조림,깍두기


In [18]:
pd.DataFrame(allergy_free_diets[('우유','밀')]).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,찰현미밥,감자탕,순살닭볶음탕,쌀밥,바지락국,오이토마토샐러드(발사믹드레싱),참나물볶음,나박김치,수수밥,북어국,소고기산적,애호박볶음,파김치
1,차조밥,소고기콩나물국,메추리알고구마조림,새우편마늘볶음밥,무채국,주꾸미채소볶음,empty,비트초절이,고구마밥,조갯살미역국,돼지갈비찜,근대나물무침,나박김치
2,소고기가지덮밥,표고버섯탕,돼지고기찹쌀구이,쌀밥,두부대합탕국,건새우애호박볶음,애호박볶음,열무물김치,율무밥,소갈비탕,달걀말이,무말랭이,깍두기
3,잡곡밥,애호박맑은국,소불고기,소고기가지덮밥,느타리버섯국,닭가슴살데리야끼조림,도라지무침,오이피클,흑미밥,낙지연포탕,우엉채곤약조림,empty,비트초절이
4,가지채소볶음밥,두부대합탕국,가지나물,율무밥,감자양파국,양상추유자청무침,양상추유자청무침,비트초절이,수수밥,모듬버섯찌개,오리훈제채소구이,empty,열무물김치


In [19]:
pd.DataFrame(allergy_free_diets[('난류','대두')]).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,현미밥,모듬버섯찌개,양념가지찜,소고기양배추덮밥,느타리버섯국,순살닭볶음탕,참나물볶음,나박김치,수수밥,감자탕,고등어구이,애호박볶음,파김치
1,보리밥,실파맑은국,두부조림,새우편마늘볶음밥,무채국,주꾸미채소볶음,empty,비트초절이,고구마밥,팽이버섯맑은국,돼지갈비찜,근대나물무침,나박김치
2,소고기가지덮밥,버섯맑은국,감자채볶음,찹쌀밥,시금치맑은국,건새우애호박볶음,애호박볶음,열무물김치,율무밥,소갈비탕,소고기찹스테이크,애호박나물,나박김치
3,잡곡밥,애호박맑은국,고등어조림,소고기가지덮밥,느타리버섯국,닭가슴살데리야끼조림,도라지무침,오이피클,흑미밥,낙지연포탕,우엉채곤약조림,empty,비트초절이
4,가지채소볶음밥,감자국,가지나물,율무밥,감자양파국,애호박당근볶음,empty,비트초절이,소고기잡채덮밥,모듬버섯찌개,오리훈제채소구이,토마토카프레제,열무물김치


In [20]:
a = pd.DataFrame(allergy_free_diets[('난류','대두')])
len([i for i in a.to_numpy() if '</s>' in i]) / len(a)

0.03378582202111614

In [38]:
with open('/home/siun97/Diet/KDD2025/AGGBERT/generated_diets/RF_diets.pickle', 'wb') as fw:
    pickle.dump(allergy_free_diets, fw)

In [18]:
len([i for i in a.to_numpy() if '</s>' in i]) / len(a)

0.035897435897435895

### GRAPH EMBEDDING in decoding

In [22]:
#2) With Graph Embedding

duplicates =[]
mps_scores =[]
ingre_sim_score = []
success_rate =[]

for a in allergy_conditions:
    final_diet = allergy_free_diets[tuple(a)]
    
    total_num_duplicates, total_mps_score, total_ingredient_cosine_sim, total_success_rate = eval_diet(final_diet, nutrient_df, a, food_columns_dict, morning_comb, lunch_comb, dinner_comb, vocab)
    print("allergy: {}".format(a), "total_num_duplicates: {:.4f}".format(total_num_duplicates), "total_mps_score: {:.4f}".format(total_mps_score),"total_ingre_sim_score: {:.4f}".format(total_ingredient_cosine_sim),  'total_success_rate: {:.4f}'.format(total_success_rate))
    
    duplicates.append(total_num_duplicates)
    mps_scores.append(total_mps_score)
    ingre_sim_score.append(total_ingredient_cosine_sim)
    success_rate.append(total_success_rate)

allergy: ['난류'] total_num_duplicates: 0.1170 total_mps_score: 0.0254 total_ingre_sim_score: 0.7359 total_success_rate: 0.9689
allergy: ['우유'] total_num_duplicates: 0.1110 total_mps_score: 0.0244 total_ingre_sim_score: 0.7360 total_success_rate: 0.9955
allergy: ['밀'] total_num_duplicates: 0.1089 total_mps_score: 0.0249 total_ingre_sim_score: 0.7363 total_success_rate: 0.9925
allergy: ['대두'] total_num_duplicates: 0.1149 total_mps_score: 0.0256 total_ingre_sim_score: 0.7361 total_success_rate: 0.9707
allergy: ['땅콩+견과류'] total_num_duplicates: 0.1074 total_mps_score: 0.0252 total_ingre_sim_score: 0.7355 total_success_rate: 1.0000
allergy: ['난류', '우유'] total_num_duplicates: 0.1143 total_mps_score: 0.0264 total_ingre_sim_score: 0.7359 total_success_rate: 0.9653
allergy: ['난류', '밀'] total_num_duplicates: 0.1137 total_mps_score: 0.0272 total_ingre_sim_score: 0.7362 total_success_rate: 0.9653
allergy: ['난류', '대두'] total_num_duplicates: 0.1237 total_mps_score: 0.0270 total_ingre_sim_score: 0.7345

In [23]:
np.mean(duplicates[:5]), np.mean(duplicates[5:15]), np.mean(duplicates[15:25]), np.mean(duplicates[25:30]), np.mean(duplicates[-1])

(0.1118552036199095,
 0.11469079939668177,
 0.1200603318250377,
 0.11625942684766215,
 0.12368024132730016)

In [24]:
np.mean(mps_scores[:5]), np.mean(mps_scores[5:15]), np.mean(mps_scores[15:25]), np.mean(mps_scores[25:30]), np.mean(mps_scores[-1])

(0.025116602854159453,
 0.026100475693235918,
 0.027330316742081508,
 0.02640213481842446,
 0.028147116834899705)

In [25]:
np.mean(ingre_sim_score[:5]), np.mean(ingre_sim_score[5:15]), np.mean(ingre_sim_score[15:25]), np.mean(ingre_sim_score[25:30]), np.mean(ingre_sim_score[-1])

(0.7359846311178885,
 0.7357228891364387,
 0.7349672513627089,
 0.735286859710382,
 0.7335971703983247)

In [26]:
np.mean(success_rate[:5]), np.mean(success_rate[5:15]), np.mean(success_rate[15:25]), np.mean(success_rate[25:30]), np.mean(success_rate[-1])

(0.9855203619909503,
 0.9723076923076924,
 0.9549321266968326,
 0.9667571644042232,
 0.9438914027149321)

In [29]:
with open('/home/siun97/Diet/KDD2025/AGGBERT/generated_diets/RL+Graph_diets.pickle', 'wb') as fw:
    pickle.dump(allergy_free_diets, fw)

In [27]:
pd.read_csv(test_datapath).head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,밤찹쌀팥밥,감자탕,순살닭볶음탕,쌀밥,바지락된장국,참나물볶음,코다리엿장조림,나박김치,수수밥,팥죽,장산적조림,갈치카레튀김,파김치
1,보리밥,미역만두국,양념가지찜,돼지고기양배추덮밥,느타리버섯국,오이토마토샐러드(발사믹드레싱),매콤콩나물무침,나박김치,밤찹쌀팥밥,만두닭개장,empty,애호박볶음,깻잎김치
2,소고기가지덮밥,모듬버섯찌개,우엉채곤약조림,밤콩밥,만두국,애호박건새우볶음,매생이육전,깍두기,수수밥,애호박맑은국,감자채전,팽이버섯나물,깻잎김치
3,잡곡밥,실파콩나물국,달걀새송이장조림,새우편마늘볶음밥,맑은가쓰오부시장국,돼지고기짜장볶음,empty,비트초절이,고구마밥,유부팽이국,감자돼지갈비찜,깻잎새콤무침,나박김치
4,가지채소볶음밥,소고기콩나물국,메추리알고구마조림,밤콩밥,애호박두부젓국,주꾸미채소볶음,미나리초무침,나박김치,밤찹쌀팥밥,감자탕,해물김치전,근대나물무침,총각김치
5,팥밥,채소스프,두부채소굴소스볶음,밤콩밥,김치두부국,단배추멸치조림,팽이버섯나물,열무물김치,두부카레라이스,실파김국,더덕사과무침,양상추샐러드(오리엔탈드레싱),비트초절이
6,고구마밥,굴두부국,돼지고기찹쌀구이,닭갈비덮밥,새우살시금치국,단호박해물찜,양배추샐러드(레몬드레싱),나박김치,밤찹쌀팥밥,소갈비탕,양배추달걀말이,애호박양파볶음,깍두기
7,콩밥,떡국떡미역국,팽이버섯나물,두부카레라이스,두부대합탕국,느타리버섯당근볶음,애호박볶음,비트초절이,율무밥,애호박두부된장국,empty,감자그라탕,나박김치
8,검정콩밥,실파콩나물국,감자채볶음,찹쌀밥,맑은가쓰오부시장국,양배추샐러드(레몬드레싱),참나물겉절이,열무물김치,장조림버터비빔밥,두부장국,부추달걀볶음,건새우두부조림,석박지
9,쌀밥,애호박맑은국,견과류고구마범벅,밤콩밥,느타리버섯국,도라지무침,닭가슴살데리야끼조림,오이피클,소고기굴소스볶음밥,만두국,우엉채곤약조림,empty,비트초절이


In [28]:
pd.DataFrame(allergy_free_diets[('난류', '대두')]).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,현미밥,모듬버섯찌개,양념가지찜,소고기양배추덮밥,느타리버섯국,순살닭볶음탕,참나물볶음,나박김치,수수밥,감자탕,고등어구이,애호박볶음,파김치
1,보리밥,실파맑은국,소고기장조림,새우편마늘볶음밥,무채국,주꾸미채소볶음,empty,비트초절이,고구마밥,팽이버섯맑은국,돼지갈비찜,근대나물무침,나박김치
2,소고기가지덮밥,버섯맑은국,감자채볶음,찹쌀밥,시금치맑은국,건새우애호박볶음,애호박볶음,열무물김치,율무밥,소갈비탕,소고기찹스테이크,애호박나물,나박김치
3,잡곡밥,애호박맑은국,고등어조림,소고기가지덮밥,느타리버섯국,닭가슴살데리야끼조림,도라지무침,오이피클,흑미밥,낙지연포탕,우엉채곤약조림,empty,비트초절이
4,가지채소볶음밥,감자국,가지나물,율무밥,감자양파국,애호박당근볶음,empty,비트초절이,소고기잡채덮밥,모듬버섯찌개,오리훈제채소구이,토마토카프레제,열무물김치


In [26]:
pd.DataFrame(allergy_free_diets[('우유', '대두')]).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,찹쌀밥,모듬버섯찌개,양념가지찜,돼지불고기덮밥,느타리버섯국,참나물볶음,오이토마토샐러드(발사믹드레싱),깍두기,찰현미밥,애호박맑은국,소고기장조림,애호박볶음,배추김치
1,보리밥,소고기미역국,순살찜닭,새우편마늘볶음밥,맑은미역국,주꾸미채소볶음,empty,비트초절이,고구마밥,팽이버섯맑은국,감자돼지갈비찜,깻잎새콤무침,나박김치
2,소고기가지덮밥,시금치북어국,감자채볶음,찹쌀밥,동태맑은국,양배추샐러드(레몬드레싱),참나물겉절이,비트초절이,율무밥,소갈비탕,소고기찹스테이크,애호박양파볶음,깍두기
3,잡곡밥,애호박맑은국,삼치구이,소고기가지덮밥,느타리버섯국,닭가슴살데리야끼조림,도라지무침,오이피클,수수밥,낙지연포탕,우엉채곤약조림,empty,비트초절이
4,가지채소볶음밥,두부대합탕국,가지나물,율무밥,감자양파국,애호박당근볶음,양상추유자청무침,콜라비나박김치,단호박영양밥,모듬버섯찌개,오리훈제채소구이,empty,열무물김치


In [28]:
a = pd.DataFrame(allergy_free_diets[('난류','대두')])
len([i for i in a.to_numpy() if '</s>' in i]) / len(a)

0.05399698340874812

In [104]:
## 수집된 전체 데이터:

# test dataset 다시 평가

#total_diet = pd.read_csv("/home/siun97/Diet/KDD2025/AGGBERT/data/total_diet.csv")
total_diet = pd.read_csv(test_datapath)
total_diet = total_diet.to_numpy()

In [105]:
len(total_diet)

836

In [106]:
def eval_diet_for_original(sequences, composition_class_dict, incidence_matrix, vocab):
    '''
    sequece : 2d list of sequence (string) [num_sequences, seq_len]
    '''
    total_num_duplicates = 0 # 낮을 수록 좋음
    total_mps_score = 0 # 낮을 수록 좋음
    total_beta_score = 0 # 높을 수록 좋음
    #total_rdi_score = 0 # 높을 수록 좋음

    for s_idx, seq in enumerate(sequences):
        # [1] check duplication (composition 3) <- 식단 내에 중복 음식이 있는 가? (없어야 함)
        temp_seq = [token for idx, token in enumerate(seq) if idx not in [0,3,7,8,12] ] # 밥 위치 제외
        if len(temp_seq) != len(set(temp_seq)): # 0,5,12
            total_num_duplicates += 1
        # [2] check location (composition 1) <- 기존 식단 작성 규칙에 위배 되는 가? (0점이어야 함)
        mps_score = calculate_misposition_score(seq, composition_class_dict)
        if mps_score != 0:
            total_mps_score += 1
        
           # print(seq)

        # [3] check incidence (composition 2) <- 기존 식단의 작성 분포에 위배 되는 가? (17점 만점)
        beta_score = calculate_beta_score(seq, incidence_matrix, vocab)
        total_beta_score += beta_score
        
    total_num_duplicates /= len(sequences)
    total_mps_score /= len(sequences)
    total_beta_score /= len(sequences)
    #total_rdi_score /= len(sequences)
    #total_success_rate /= len(sequences)

    return total_num_duplicates, total_mps_score, total_beta_score

def allergy_check_original(sequences, allergy_reference, allery_name):
    '''
    sequece : 2d list of sequence (string) [num_sequences, seq_len]
    '''
    total_success_rate = 0

    for s_idx, seq in enumerate(sequences):

        allergy_rate = allergy_free_test(seq, allergy_reference, allery_name)
        if allergy_rate == 0:
            total_success_rate += 1
    
    total_success_rate /= len(sequences)

    return total_success_rate

In [107]:
def calculate_misposition_score(sample_diet, composition_class_dict):
    
    diet_pos = []
    for token in sample_diet:
        if token == '[MASK]' or token == '<s>' or token == '</s>':
            diet_pos.append(token)
        else:
            composition_info = composition_class_dict[token]
            nutri = list(composition_info.index[composition_info == 1].values)     
            diet_pos.append(nutri)
    
    mis_pos = 0
        
    if not any(label in diet_pos[0] for label in ['rice', 'special']):
            mis_pos += 1
    if not any(label in diet_pos[1] for label in ['soup', 'empty']):
            mis_pos += 1
    if not any(label in diet_pos[2] for label in ['side_dish', 'kimchi']):
            mis_pos += 1
    if not any(label in diet_pos[3] for label in ['rice','special']):
            mis_pos += 1
    if not any(label in diet_pos[4] for label in ['soup', 'empty']):
            mis_pos += 1
    if not any(label in diet_pos[5] for label in ['side_dish']):
            mis_pos += 1
    if not any(label in diet_pos[6] for label in ['side_dish', 'empty']):
            mis_pos += 1
    if not any(label in diet_pos[7] for label in ['kimchi']):
            mis_pos += 1
    if not any(label in diet_pos[8] for label in ['rice','special']):
            mis_pos += 1
    if not any(label in diet_pos[9] for label in ['soup', 'empty']):
            mis_pos += 1
    if not any(label in diet_pos[10] for label in ['side_dish']):
            mis_pos += 1
    if not any(label in diet_pos[11] for label in ['side_dish']):
            mis_pos += 1
    if not any(label in diet_pos[12] for label in ['kimchi']):
            mis_pos += 1
    
    return mis_pos

In [108]:

total_num_duplicates, total_mps_score, total_beta_score = eval_diet_for_original(total_diet, category_dict, incidence_matrix, vocab)
    
print("total_num_duplicates: {:.4f}".format(total_num_duplicates), "total_beta_score: {:.4f}".format(total_beta_score), "total_mps_score: {:.4f}".format(total_mps_score))

for a in allergy_conditions:
    total_success_rate = allergy_check_original(total_diet, nutrient_df, a)
    print("allergy: {}".format(a), 'total_success_rate: {:.4f}'.format(total_success_rate))

total_num_duplicates: 0.0275 total_beta_score: 1.1435 total_mps_score: 0.1974
allergy: ['난류'] total_success_rate: 0.4079
allergy: ['우유'] total_success_rate: 0.7919
allergy: ['밀'] total_success_rate: 0.4306
allergy: ['대두'] total_success_rate: 0.3433
allergy: ['땅콩+견과류'] total_success_rate: 0.8888
allergy: ['난류', '우유'] total_success_rate: 0.3696
allergy: ['난류', '밀'] total_success_rate: 0.2632
allergy: ['난류', '대두'] total_success_rate: 0.1818
allergy: ['난류', '땅콩+견과류'] total_success_rate: 0.3553
allergy: ['우유', '밀'] total_success_rate: 0.3864
allergy: ['우유', '대두'] total_success_rate: 0.2967
allergy: ['우유', '땅콩+견과류'] total_success_rate: 0.7117
allergy: ['밀', '대두'] total_success_rate: 0.2524
allergy: ['밀', '땅콩+견과류'] total_success_rate: 0.3816
allergy: ['대두', '땅콩+견과류'] total_success_rate: 0.3122
allergy: ['난류', '우유', '밀'] total_success_rate: 0.2464
allergy: ['난류', '우유', '대두'] total_success_rate: 0.1651
allergy: ['난류', '우유', '땅콩+견과류'] total_success_rate: 0.3230
allergy: ['난류', '밀', '대두'] total_s